Data Cleaning

In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords

# Download required NLTK data
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# List of MBTI types to remove
mbti_types = ['infj', 'entp', 'intp', 'intj', 'entj', 'enfj', 'infp', 'enfp',
              'isfp', 'istp', 'isfj', 'istj', 'estp', 'esfp', 'estj', 'esfj']

# 1. Load the Dataset
file_path = 'mbti_1.csv'
df = pd.read_csv(file_path, engine='python', on_bad_lines='skip')

print("Dataset Shape:", df.shape)
print(df.head())

# 2. Define the Cleaning Function
def clean_text(text):
    # 1. Lowercase
    text = text.lower()

    # 2. Explicitly remove the ||| separator
    text = text.replace('|||', ' ')

    # 3. Remove URLs
    text = re.sub(r'https?://[^\s<>"]+|www\.[^\s<>"]+', ' ', text)

    # 4. Remove Punctuation & Numbers (keep only letters)
    text = re.sub(r'[^a-z\s]', ' ', text)

    # 5. Tokenize (split into words) and remove stop words & MBTI mentions
    words = text.split()
    cleaned_words = [word for word in words if word not in stop_words and word not in mbti_types]

    text = ' '.join(cleaned_words)

    return text

# 3. Apply the Cleaning Function to the DataFrame
print("Cleaning data... this might take a moment.")
df['cleaned_posts'] = df['posts'].apply(clean_text)
df = df[df['cleaned_posts'].str.strip() != '']

# Check the results
print(df[['type', 'cleaned_posts']].head(4))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Dataset Shape: (8675, 2)
   type                                              posts
0  INFJ  'http://www.youtube.com/watch?v=qsXHcwe3krw|||...
1  ENTP  'I'm finding the lack of me in these posts ver...
2  INTP  'Good one  _____   https://www.youtube.com/wat...
3  INTJ  'Dear INTP,   I enjoyed our conversation the o...
4  ENTJ  'You're fired.|||That's another silly misconce...
Cleaning data... this might take a moment.
   type                                      cleaned_posts
0  INFJ  moments sportscenter top ten plays pranks life...
1  ENTP  finding lack posts alarming sex boring positio...
2  INTP  good one course say know blessing curse absolu...
3  INTJ  dear enjoyed conversation day esoteric gabbing...


TF-IDF Vectorization

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Separate Features (X) and Labels (y)
X = df['cleaned_posts']
y = df['type']

# 2. Split the Data
# test_size=0.2 means 80% data for training, 20% for testing.
# stratify=y is VERY important here. It ensures your train/test split has the
# same percentage of each of the 16 MBTI types (preventing imbalanced splits).
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training data size: {X_train.shape[0]} rows")
print(f"Testing data size: {X_test.shape[0]} rows")

# 3. Initialize TF-IDF Vectorizer
# max_features=5000 limits the vocabulary to the top 5,000 words.
# This prevents your matrix from becoming too massive and crashing your RAM.
tfidf = TfidfVectorizer(max_features=5000)

# 4. Fit and Transform
print("Vectorizing text... this takes a few seconds.")

# fit_transform Learns the vocabulary from the training data AND turns it into numbers
X_train_tfidf = tfidf.fit_transform(X_train)

# transform ONLY turns text into numbers based on the vocabulary learned above.
# We NEVER 'fit' on testing data!
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF Vectorization Complete!")
print(f"Training Matrix Shape: {X_train_tfidf.shape}")

Training data size: 6939 rows
Testing data size: 1735 rows
Vectorizing text... this takes a few seconds.
TF-IDF Vectorization Complete!
Training Matrix Shape: (6939, 5000)


Model Training

In [ ]:
#v3
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib

print("1. Initializing Logistic Regression Model...")
# C is the regularization strength. Lower values prevent overfitting.
# max_iter is increased because 5000 features take longer to converge.
log_model = LogisticRegression(
    C=1.0,
    class_weight='balanced',    # Forces model to pay attention to minority classes
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

print("2. Training the linear model (This should be much faster!)...")
log_model.fit(X_train_tfidf, y_train)
print("Training Complete!\n")

print("3. Evaluating the model...")
y_pred = log_model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Model Accuracy: {accuracy * 100:.2f}%\n")

print("Classification Report:")
print("-" * 60)
print(classification_report(y_test, y_pred))

print("\n4. Saving models to Colab storage...")
joblib.dump(log_model, 'log_reg_mbti_model.joblib')
joblib.dump(tfidf, 'tfidf_vectorizer.joblib')
print("Models saved! You can download them manually.")

1. Initializing Logistic Regression Model...
2. Training the linear model (This should be much faster!)...
Training Complete!

3. Evaluating the model...
Overall Model Accuracy: 48.82%

Classification Report:
------------------------------------------------------------
              precision    recall  f1-score   support

        ENFJ       0.26      0.53      0.34        38
        ENFP       0.48      0.48      0.48       135
        ENTJ       0.21      0.46      0.28        46
        ENTP       0.53      0.46      0.49       137
        ESFJ       0.20      0.25      0.22         8
        ESFP       0.00      0.00      0.00        10
        ESTJ       0.33      0.12      0.18         8
        ESTP       0.15      0.22      0.18        18
        INFJ       0.63      0.43      0.51       294
        INFP       0.67      0.55      0.60       366
        INTJ       0.55      0.45      0.50       218
        INTP       0.60      0.58      0.59       261
        ISFJ       0.27    

Test

In [4]:
import pandas as pd

# 1. Sample Friends Dialogues (A mix of iconic personality traits)
# You can change these quotes to test different characters!
test_dialogues = [
    "Could I BE wearing any more clothes? I'm finding the lack of sarcasm in this room alarming. I use humor as a defense mechanism.", # Chandler
    "Rules control the fun! I need everything to be perfectly organized and clean, otherwise I can't relax. I just care so much!", # Monica
    "How you doin'? Look, I'm not great at the advice. Can I interest you in a sarcastic comment? Some cheese? I just want to eat pizza.", # Joey
    "We were on a break! Look, paleontology is a serious science. I need facts, logic, and for people to respect my dinosaur models." # Ross
]

characters = ["Chandler", "Monica", "Joey", "Ross"]

print("Preprocessing test data...\n")

# 2. Clean the test dialogues using the 'clean_text' function you wrote in Phase 1
cleaned_dialogues = [clean_text(text) for text in test_dialogues]

# 3. Vectorize the text
# IMPORTANT: We only use .transform() on new data. We never .fit() again!
dialogues_tfidf = tfidf.transform(cleaned_dialogues)

# 4. Make predictions
predictions = log_model.predict(dialogues_tfidf)

# 5. Get prediction probabilities (How confident is the model?)
probabilities = log_model.predict_proba(dialogues_tfidf)
classes = log_model.classes_

# 6. Display the Results
print("--- Friends Character MBTI Predictions ---")
for i in range(len(test_dialogues)):
    print(f"Character: {characters[i]}")
    print(f"Sample Quote: '{test_dialogues[i]}'")
    print(f"Predicted MBTI: ** {predictions[i]} **")

    # Find the confidence percentage for the top prediction
    top_prob_index = list(classes).index(predictions[i])
    confidence = probabilities[i][top_prob_index] * 100
    print(f"Model Confidence: {confidence:.2f}%\n")

Preprocessing test data...

--- Friends Character MBTI Predictions ---
Character: Chandler
Sample Quote: 'Could I BE wearing any more clothes? I'm finding the lack of sarcasm in this room alarming. I use humor as a defense mechanism.'
Predicted MBTI: ** INTJ **
Model Confidence: 17.40%

Character: Monica
Sample Quote: 'Rules control the fun! I need everything to be perfectly organized and clean, otherwise I can't relax. I just care so much!'
Predicted MBTI: ** ISTP **
Model Confidence: 17.79%

Character: Joey
Sample Quote: 'How you doin'? Look, I'm not great at the advice. Can I interest you in a sarcastic comment? Some cheese? I just want to eat pizza.'
Predicted MBTI: ** INTJ **
Model Confidence: 14.09%

Character: Ross
Sample Quote: 'We were on a break! Look, paleontology is a serious science. I need facts, logic, and for people to respect my dinosaur models.'
Predicted MBTI: ** INTJ **
Model Confidence: 21.20%

